In [2]:
# Cell 0 — Setup
from pathlib import Path
import sys, json
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")


True

In [6]:

# Cell 1 — Build pipeline + evaluator
from src.retrieval import HybridRetriever
from src.generation import Generator
from src.pipeline import RAGPipeline
from src.evaluation import (
    TestQuestion, LLMJudge, Evaluator, summarize_results, save_results
)

retriever = HybridRetriever(
    chroma_path=str(PROJECT_ROOT / "chroma_db"),
    collection_name="financial_risk_10k",
)
generator = Generator(model_name="claude-opus-5-5")
pipeline = RAGPipeline(retriever=retriever, generator=generator, top_k=5)
judge = LLMJudge(model_name="claude-opus-5-5")  # Bigger model for judging

evaluator = Evaluator(pipeline=pipeline, judge=judge, delay_seconds=5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading chunks from collection 'financial_risk_10k'...
Building BM25 index over 1401 chunks...
✓ HybridRetriever ready: 1401 chunks


In [7]:

# Cell 2 — Load test set
test_data = json.loads((PROJECT_ROOT / "data" / "evaluation_test_set"/"test_set.json").read_text())
questions = [TestQuestion(**row) for row in test_data]
print(f"Loaded {len(questions)} test questions")


Loaded 18 test questions


In [8]:

# Cell 3 — Run evaluation
# At ~15s per question (with rate limiting), 25 questions ≈ 6-8 minutes
results = evaluator.evaluate_all(questions)


[1/18] single_01: What cybersecurity risks does JPMorgan disclose in its Risk ...
  P@5=0.80  R@5=1.00  faith=4  rel=3
[2/18] single_02: How does Capital One describe its commercial real estate exp...
  P@5=0.60  R@5=1.00  faith=5  rel=4
[3/18] single_03: What climate-related risks does BlackRock identify?...
  P@5=1.00  R@5=1.00  faith=4  rel=4
[4/18] single_04: What are PayPal's main risks from third-party service provid...
  P@5=0.20  R@5=1.00  faith=4  rel=4
[5/18] single_05: What interest rate risks does Goldman Sachs disclose?...
  P@5=0.20  R@5=1.00  faith=3  rel=1
[6/18] single_06: What geopolitical risks does Visa identify?...
  P@5=0.40  R@5=1.00  faith=5  rel=5
[7/18] comparison_01: How do JPM and GS describe their stress-testing and capital ...
  P@5=0.20  R@5=0.50  faith=4  rel=3
[8/18] comparison_02: Compare the cybersecurity risk disclosures of Mastercard and...
  P@5=0.60  R@5=1.00  faith=3  rel=5
[9/18] comparison_03: How do BAC and COF differ in how they describe cons


###### Evaluation after adding reranking

[1/18] single_01: What cybersecurity risks does JPMorgan disclose in its Risk ...
  P@5=0.80  R@5=1.00  faith=4  rel=3
[2/18] single_02: How does Capital One describe its commercial real estate exp...
  P@5=0.60  R@5=1.00  faith=5  rel=4
[3/18] single_03: What climate-related risks does BlackRock identify?...
  P@5=1.00  R@5=1.00  faith=4  rel=4
[4/18] single_04: What are PayPal's main risks from third-party service provid...
  P@5=0.20  R@5=1.00  faith=4  rel=4
[5/18] single_05: What interest rate risks does Goldman Sachs disclose?...
  P@5=0.20  R@5=1.00  faith=3  rel=1
[6/18] single_06: What geopolitical risks does Visa identify?...
  P@5=0.40  R@5=1.00  faith=5  rel=5
[7/18] comparison_01: How do JPM and GS describe their stress-testing and capital ...
  P@5=0.20  R@5=0.50  faith=4  rel=3
[8/18] comparison_02: Compare the cybersecurity risk disclosures of Mastercard and...
  P@5=0.60  R@5=1.00  faith=3  rel=5
[9/18] comparison_03: How do BAC and COF differ in how they describe consumer cred...
  P@5=0.60  R@5=1.00  faith=4  rel=4
[10/18] comparison_04: How do PYPL and V describe their exposure to cryptocurrency ...
  P@5=1.00  R@5=1.00  faith=4  rel=5
[11/18] comparison_05: How do AXP and BLK describe their climate-related risks?...
  P@5=0.40  R@5=1.00  faith=4  rel=4
[12/18] specific_01: Which filings discuss CCAR stress testing requirements?...
  P@5=0.60  R@5=0.50  faith=3  rel=4
[13/18] specific_02: Which companies disclose risks related to Dodd-Frank Act req...
  P@5=1.00  R@5=0.40  faith=2  rel=3
[14/18] specific_03: Which companies mention stablecoin in their Risk Factors?...
  P@5=0.80  R@5=0.57  faith=2  rel=3
[15/18] paraphrase_01: What dangers do these banks face from people breaking into t...
  P@5=1.00  R@5=0.44  faith=3  rel=4
[16/18] paraphrase_02: What problems could arise from companies these firms hire to...
  P@5=1.00  R@5=0.56  faith=4  rel=2
[17/18] out_of_scope_01: What was JPMorgan's exact net income for fiscal year 2024?...
  P@5=0.00  R@5=1.00  faith=None  rel=None
[18/18] out_of_scope_02: Which bank has the best customer service?...
  P@5=0.00  R@5=1.00  faith=None  rel=None

In [9]:

# Cell 4 — Summary
summary = summarize_results(results)
print(json.dumps(summary, indent=2))


{
  "n_questions": 18,
  "retrieval": {
    "avg_precision_at_5": 0.5777777777777778,
    "avg_recall_at_5": 0.8317460317460318,
    "avg_mrr": 0.7916666666666666
  },
  "generation": {
    "avg_faithfulness": 3.625,
    "avg_relevance": 3.625,
    "citation_hallucinations": 0,
    "refusal_accuracy": 0.8333333333333334
  },
  "avg_latency_ms": 6963.781833648682
}


In [10]:

# Cell 5 — Save for future comparison
save_results(results, str(PROJECT_ROOT / "data" / "eval_run_baseline.json"))


✓ Saved 18 results to e:\P1_JOB\AI projects\financial_risk_rag\data\eval_run_baseline.json


In [12]:

# Cell 6 — Inspect failures
print("\n=== FAILURES TO INVESTIGATE ===")
for r in results:
    fails = []
    if r.precision_at_5 < 0.5:
        fails.append(f"P@5={r.precision_at_5:.2f}")
    if r.faithfulness_score and r.faithfulness_score <= 3:
        fails.append(f"faith={r.faithfulness_score}")
    if r.citation_hallucination:
        fails.append("cite_halluc")
    if r.refusal_correct is False:
        fails.append("wrong_refusal")
    if fails:
        print(f"\n{r.question_id} [{r.category}]: {', '.join(fails)}")
        print(f"  Q: {r.question}")
        print(f"  Retrieved: {r.retrieved_tickers}")
        print(f"  Answer: {r.answer[:200]}...")


=== FAILURES TO INVESTIGATE ===

single_04 [single_company]: P@5=0.20
  Q: What are PayPal's main risks from third-party service providers?
  Retrieved: ['PYPL', 'COF', 'COF', 'AXP', 'AXP']
  Answer: PayPal relies on third parties across many parts of its business [Chunk 1]:

- **Transaction processing:** networks, banks, payment processors, and payment gateways that link it to the payment card an...

single_05 [single_company]: P@5=0.20, faith=3, wrong_refusal
  Q: What interest rate risks does Goldman Sachs disclose?
  Retrieved: ['COF', 'GS', 'AXP', 'COF', 'JPM']
  Answer: The provided excerpts do not contain sufficient information to answer this question.

The only Goldman Sachs excerpt covers conflicts of interest and improper information sharing, not interest rate ri...

single_06 [single_company]: P@5=0.40
  Q: What geopolitical risks does Visa identify?
  Retrieved: ['V', 'JPM', 'BLK', 'BAC', 'V']
  Answer: Visa identifies several geopolitical risks that could affect its busin

In [11]:
# Search for LIBOR or SOFR in chunks
hits = [
    (m.get("ticker"), text[:200])
    for m, text in zip(retriever.chunk_metas, retriever.chunk_texts)
    if "LIBOR" in text or "SOFR" in text
]
print(f"Chunks mentioning LIBOR/SOFR: {len(hits)}")
for ticker, snippet in hits[:5]:
    print(f"\n[{ticker}] {snippet}...")

Chunks mentioning LIBOR/SOFR: 0
